In [ ]:
# %pip install -U langchain-chroma

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.prompts import PromptTemplate

# from langchain_community.vectorstores import Chroma   
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader

from langchain_classic.chains import RetrievalQA
# import gradio as gr

In [4]:
import os
from dotenv import load_dotenv

# .env파일의 환경변수를 불러옵니다.
load_dotenv()

# 환경 변수에서 API KEY를 가져옵니다.
api_key = os.getenv("OPENAI_API_KEY")


In [5]:
os.environ['OPENAI_API_KEY'] =  api_key

In [ ]:
# import urllib.request

# urllib.request.urlretrieve("https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch07/2020_경제금융용어 700선_게시.pdf", filename="2020_경제금융용어 700선_게시.pdf")

In [6]:
loader = PyPDFLoader("2020_경제금융용어 700선_게시.pdf")
texts = loader.load_and_split()

In [ ]:
print('문서의 수 :', len(texts))

In [ ]:
texts[15]

In [ ]:
print(texts[15].page_content)

In [ ]:
# 0번 문서는 머리말
print(texts[0].page_content)

In [ ]:
print(texts[5].page_content)

In [ ]:
# 12번 문서까지는 목차
print(texts[12].page_content)

In [ ]:
# 13번 문서부터는 금융 용어 설명
print(texts[13].page_content)

In [ ]:
texts = texts[13:]
print('줄어든 texts의 길이 :', len(texts))

In [ ]:
print('첫번째 문서 출력 :', texts[0])

In [ ]:
print(texts[-1])

In [ ]:
print(texts[-2])

In [ ]:
# 마지막 데이터를 제거
texts = texts[:-1]
print('마지막 데이터 제거 후 texts의 길이 :', len(texts))

In [ ]:
print('마지막 데이터 출력')
texts[-1]

In [ ]:
# 벡터 데이터베이스를 생성합니다.
embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embedding)

In [ ]:
# 벡터DB의 개수 확인
vectordb._collection.count()

In [ ]:
# 메타 정보 확인
for key in vectordb._collection.get():
  print(key)

In [ ]:
documents = vectordb._collection.get()["documents"]
print("청크의 갯수 :", len(documents))
print("-" * 50)
print("청크의 내용 :", documents[0])

In [ ]:
# embedding vetor만 조회하기
embeddings = vectordb._collection.get(include=['embeddings'])['embeddings']
print('임베딩 벡터의 개수 :', len(embeddings))

In [ ]:
print('첫번째 문서의 임베딩 값 출력 :', embeddings[0])
print('첫번째 문서의 임베딩 값의 길이 :', len(embeddings[0]))

In [ ]:
metadatas = vectordb._collection.get()['metadatas']
print('metadatas의 개수 :', len(metadatas))
print('첫번째 문서의 출처 :', metadatas[0])

In [ ]:
# 유사도가 높은 문서 2개만 추출. k = 2
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [ ]:
docs = retriever.invoke("비트코인이 궁금해")
print('유사 문서 개수 :', len(docs))
print('--' * 20)
print('첫번째 유사 문서 :', docs[0])
print('두번째 유사 문서 :', docs[1])

In [ ]:
# Create Prompt
template = """당신은 한국은행에서 만든 금융 용어를 설명해주는 금융쟁이입니다.
안상준 개발자가 만들었습니다. 주어진 검색 결과를 바탕으로 답변하세요.
검색 결과에 없는 내용이라면 답변할 수 없다고 하세요. 반말로 친근하게 답변하세요.
{context}

Question: {question}
Answer:
"""

prompt = PromptTemplate.from_template(template)

In [ ]:
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type_kwargs={"prompt": prompt},
    retriever=retriever,
    return_source_documents=True)

In [ ]:
input_text = "디커플링이란 무엇인가?"
chatbot_response = qa_chain.invoke(input_text)
print(chatbot_response)

In [ ]:
def get_chatbot_response(input_text):
    chatbot_response = qa_chain.invoke(input_text)
    return chatbot_response['result'].strip()

In [ ]:
input_text = "너는 누구야?"
chatbot_response = get_chatbot_response(input_text)
print(chatbot_response)

In [ ]:
input_text = "비트코인에 대해서 궁금하당~"
result = get_chatbot_response(input_text)
print(result)

In [ ]:
import gradio as gr


# =========================================================
# 챗봇 답변 처리 함수
# =========================================================
def respond(message, chat_history):

    # 빈 입력 방지
    if not message.strip():
        return "", chat_history

    # 기존 history가 None이면 빈 리스트로 초기화
    if chat_history is None:
        chat_history = []

    # RAG 챗봇 답변 생성
    bot_message = get_chatbot_response(message)

    # Gradio messages 형식으로 대화 기록 추가
    chat_history.append(
        {
            "role": "user",
            "content": message
        }
    )

    chat_history.append(
        {
            "role": "assistant",
            "content": bot_message
        }
    )

    # 입력창 비우기 + 대화 기록 반환
    return "", chat_history


# =========================================================
# Gradio UI
# =========================================================
with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 💰 경제금융용어 챗봇
        한국은행 『경제금융용어 700선』을 기반으로 답변합니다.
        """
    )

    # 현재 설치 버전에서는 type 파라미터 제거
    chatbot = gr.Chatbot(
        label="경제금융용어 챗봇"
    )

    msg = gr.Textbox(
        label="질문해주세요!",
        placeholder="예: 기준금리가 뭐야?"
    )

    clear = gr.Button("대화 초기화")

    # Enter 입력 시 respond 실행
    msg.submit(
        fn=respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    # 대화 초기화
    clear.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot,
        queue=False
    )


# =========================================================
# 실행
# =========================================================
demo.launch(
    debug=True
)

In [ ]:
with gr.Blocks() as demo:

    chatbot = gr.Chatbot(
        label="경제금융용어 챗봇"
    )

    msg = gr.Textbox(
        label="질문해주세요!",
        placeholder="예: 기준금리가 뭐야?"
    )

    clear = gr.Button("대화 초기화")

    # Enter 입력 시 respond 실행
    msg.submit(
        fn=respond,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    # 대화 초기화
    clear.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot,
        queue=False
    )

    def respond(message, chat_history):

        result = qa_chain.invoke(message)

        # RAG 챗봇 답변 생성
        bot_message = result['result']
        bot_message = result['result']

        # Gradio messages 형식으로 대화 기록 추가
        if chat_history is None:
            chat_history = []

        chat_history.append({
            "role": "user",
            "content": message
        })
        
        chat_history.append({
        "role": "assistant",
        "content": bot_message
        })
        
        # 입력창 비우기 + 대화 기록 반환
        return "", chat_history

    msg.submit(respond,[msg, chatbot], [msg, chatbot])

    # 초기화 버튼을 클릭하면 채팅 기록을 초기화
    clear.click(lambda: [], None, chatbot, queue=False)

    demo.launch(
        debug=True
)

위 코드를 조금만 수정하여 챗봇을 만들 수 있습니다.